In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
import statsmodels.formula.api as smf

In [2]:
def load_and_prepare_run(
    run_id,
    result_dir=""
):
    """
    Load loss, label, FOIF and TracIn results for one experimental run.

    Expected filenames:
        IF_sp1_de01_seed_<run_id>.csv
        TC_sp1_de01_seed_<run_id>.csv
        loss_sp1_de01_seed_<run_id>.csv
        label_sp1_de01_seed_<run_id>.csv
    """

    foif_df = pd.read_csv(
        f"IF_9_1_run{run_id}_Dia.csv"
    )
    tracin_df = pd.read_csv(
        f"TC_9_1_run{run_id}_Dia.csv"
    )
    loss_df = pd.read_csv(
        f"loss_9_1_run{run_id}_Dia.csv"
    )
    label_df = pd.read_csv(
        f"Class_label_9_1_run{run_id}_Dia.csv"
    )

    label_df = label_df[
        ["id", "label"]
    ].rename(columns={"id": "Train_ID"})

    foif_df = foif_df.rename(
        columns={"Score": "FOIF_Score"}
    )

    tracin_df = tracin_df.rename(
        columns={"Score": "TracIn_Score"}
    )

    density_df = (
        label_df
        .merge(loss_df, on="Train_ID", validate="one_to_one")
        .merge(foif_df, on="Train_ID", validate="one_to_one")
        .merge(tracin_df, on="Train_ID", validate="one_to_one")
    )


    density_df["Minority"] = density_df["label"].astype(int)

    density_df["Log_Loss"] = np.log1p(
        density_df["Training_Loss"]
    )

    density_df["Log_Abs_FOIF"] = np.log1p(
        np.abs(density_df["FOIF_Score"])
    )

    density_df["Run"] = run_id

    return density_df

In [3]:
def fit_regression_for_run(density_df, run_id):
    """
    Fit signed-score and magnitude regression models for one run.
    """

    signed_model = smf.ols(
        "FOIF_Score ~ Log_Loss * Minority",
        data=density_df
    ).fit()

    magnitude_model = smf.ols(
        "Log_Abs_FOIF ~ Log_Loss * Minority",
        data=density_df
    ).fit()

    signed_result = {
        "Run": run_id,
        "Model": "Signed FOIF",
        "Beta_0_Intercept": signed_model.params["Intercept"],
        "Beta_1_Log_Loss": signed_model.params["Log_Loss"],
        "Beta_2_Minority": signed_model.params["Minority"],
        "Beta_3_Interaction": signed_model.params["Log_Loss:Minority"],
        "Minority_Slope": (
            signed_model.params["Log_Loss"]
            + signed_model.params["Log_Loss:Minority"]
        ),
    }

    magnitude_result = {
        "Run": run_id,
        "Model": "Absolute FOIF magnitude",
        "Beta_0_Intercept": magnitude_model.params["Intercept"],
        "Beta_1_Log_Loss": magnitude_model.params["Log_Loss"],
        "Beta_2_Minority": magnitude_model.params["Minority"],
        "Beta_3_Interaction": magnitude_model.params[
            "Log_Loss:Minority"
        ],
        "Minority_Slope": (
            magnitude_model.params["Log_Loss"]
            + magnitude_model.params["Log_Loss:Minority"]
        ),
    }

    return signed_result, magnitude_result

In [4]:
def calculate_decile_summary(density_df, run_id):
    """
    Calculate loss-decile statistics for one experimental run.
    """

    density_df = density_df.copy()

    density_df["Loss_Decile"] = pd.qcut(
        density_df["Training_Loss"],
        q=10,
        labels=False,
        duplicates="drop"
    ) + 1

    decile_summary = (
        density_df
        .groupby(
            ["Loss_Decile", "Minority"],
            as_index=False
        )
        .agg(
            Count=("Train_ID", "size"),

            Mean_Loss=("Training_Loss", "mean"),
            Median_Loss=("Training_Loss", "median"),

            Mean_FOIF=("FOIF_Score", "mean"),
            FOIF_Positive_Fraction=(
                "FOIF_Score",
                lambda x: (x > 0).mean()
            ),

            Mean_TracIn=("TracIn_Score", "mean"),
            TracIn_Positive_Fraction=(
                "TracIn_Score",
                lambda x: (x > 0).mean()
            )
        )
    )

    decile_summary["Run"] = run_id

    return decile_summary

In [5]:
seeds = [1,2,3,4,5]

all_regression_results = []
all_run_data = []
all_median_loss_results = []
all_decile_results = []

for seed in seeds:
    print(f"Processing run {seed}...")

    density_df = load_and_prepare_run(
        run_id=seed,
        result_dir=""
    )

    # median_loss_run = (
    # density_df
    # .groupby("Minority")["Training_Loss"]
    # .median()
    # )

    # all_median_loss_results.append({
    #     "Run": seed,
    #     "Dense_Median_Loss": median_loss_run.get("Dense", np.nan),
    #     "Sparse_Median_Loss": median_loss_run.get("Sparse", np.nan)
    # })

    decile_summary_run = calculate_decile_summary(
        density_df=density_df,
        run_id=seed
    )

    all_decile_results.append(decile_summary_run)
    

    signed_result, magnitude_result = fit_regression_for_run(
        density_df=density_df,
        run_id=seed
    )

    all_regression_results.extend([
        signed_result,
        magnitude_result
    ])

    all_run_data.append(density_df)

regression_results_df = pd.DataFrame(
    all_regression_results
)

all_density_df = pd.concat(
    all_run_data,
    ignore_index=True
)

display(regression_results_df)

Processing run 1...
Processing run 2...
Processing run 3...
Processing run 4...
Processing run 5...


,Run,Model,Beta_0_Intercept,Beta_1_Log_Loss,Beta_2_Minority,Beta_3_Interaction,Minority_Slope
0,1,Signed FOIF,0.001821,-0.080915,0.010044,0.077167,-0.003748
1,1,Absolute FOIF magnitude,0.000032,0.082821,0.008625,-0.047516,0.035305
2,2,Signed FOIF,0.001760,-0.076734,0.009860,0.084242,0.007508
3,2,Absolute FOIF magnitude,0.000093,0.079539,0.010374,-0.056549,0.022990
4,3,Signed FOIF,0.001591,-0.072875,0.008457,0.077134,0.004260
5,3,Absolute FOIF magnitude,-0.000106,0.071398,0.008330,-0.045266,0.026132
6,4,Signed FOIF,0.000924,-0.033743,0.006683,0.035241,0.001498
7,4,Absolute FOIF magnitude,0.000190,0.036623,0.005583,-0.015418,0.021204
8,5,Signed FOIF,0.001082,-0.043751,0.007845,0.047515,0.003765
9,5,Absolute FOIF magnitude,0.000136,0.048134,0.006641,-0.019760,0.028374


In [6]:
median_loss_results_df = pd.DataFrame(
    all_median_loss_results
)

display(median_loss_results_df)

""


In [7]:
coefficient_columns = [
    "Beta_0_Intercept",
    "Beta_1_Log_Loss",
    "Beta_2_Minority",
    "Beta_3_Interaction",
    "Minority_Slope",
]

regression_summary = (
    regression_results_df
    .groupby("Model")[coefficient_columns]
    .agg(["mean", "std"])
)

display(regression_summary)

Beta_0_Intercept           Beta_1_Log_Loss            \
                                    mean       std            mean       std   
Model                                                                          
Absolute FOIF magnitude         0.000069  0.000114        0.063703  0.020318   
Signed FOIF                     0.001436  0.000408       -0.061603  0.021353   

                        Beta_2_Minority           Beta_3_Interaction  \
                                   mean       std               mean   
Model                                                                  
Absolute FOIF magnitude        0.007911  0.001857          -0.036902   
Signed FOIF                    0.008578  0.001408           0.064260   

                                  Minority_Slope            
                              std           mean       std  
Model                                                       
Absolute FOIF magnitude  0.018194       0.026801  0.005502  
Signed FOIF              0.021530       0.002656  0.004174

In [8]:
decile_results_5_runs = pd.concat(
    all_decile_results,
    ignore_index=True
)

display(decile_results_5_runs)

,Loss_Decile,Minority,Count,Mean_Loss,Median_Loss,Mean_FOIF,FOIF_Positive_Fraction,Mean_TracIn,TracIn_Positive_Fraction,Run
0,1,0,727,0.000037,0.000038,0.000014,0.991747,1.864723,1.000000,1
1,1,1,73,0.000024,0.000020,0.000034,1.000000,-0.007695,0.136986,1
2,2,0,776,0.000084,0.000084,0.000027,0.979381,1.864592,1.000000,1
3,2,1,24,0.000087,0.000090,0.000113,1.000000,-0.007508,0.041667,1
4,3,0,770,0.000149,0.000148,0.000040,0.962338,1.861232,1.000000,1
...,...,...,...,...,...,...,...,...,...,...
95,8,1,105,0.007620,0.007451,0.004438,1.000000,-0.016691,0.057143,5
96,9,0,676,0.025982,0.022728,0.002748,0.986686,1.841539,1.000000,5
97,9,1,124,0.028285,0.025445,0.011314,1.000000,-0.019609,0.016129,5
98,10,0,549,0.180360,0.112103,-0.001379,0.681239,1.830157,1.000000,5


In [9]:
statistics_to_summarise = [
    "Count",
    "Mean_Loss",
    "Median_Loss",
    "Mean_FOIF",
    "FOIF_Positive_Fraction",
    "Mean_TracIn",
    "TracIn_Positive_Fraction"
]

decile_summary_5_runs = (
    decile_results_5_runs
    .groupby(
        ["Loss_Decile", "Minority"]
    )[statistics_to_summarise]
    .agg(["mean", "std"])
    .reset_index()
)

display(decile_summary_5_runs)

Loss_Decile Minority  Count            Mean_Loss           Median_Loss  \
                          mean        std      mean       std        mean   
0            1        0  727.6   6.542171  0.000035  0.000002    0.000035   
1            1        1   73.8   5.540758  0.000022  0.000001    0.000020   
2            2        0  774.2   9.471008  0.000080  0.000004    0.000079   
3            2        1   25.4   7.829432  0.000079  0.000005    0.000079   
4            3        0  771.6   3.847077  0.000141  0.000008    0.000139   
5            3        1   27.4   3.577709  0.000141  0.000008    0.000141   
6            4        0  768.8   4.604346  0.000251  0.000016    0.000246   
7            4        1   31.4   4.393177  0.000256  0.000013    0.000256   
8            5        0  762.2   3.420526  0.000485  0.000027    0.000468   
9            5        1   37.6   3.361547  0.000500  0.000030    0.000491   
10           6        0  750.4   3.911521  0.001074  0.000057    0.001049   
11           6        1   49.6   3.911521  0.001088  0.000050    0.001043   
12           7        0  728.6   8.173127  0.002784  0.000116    0.002673   
13           7        1   71.4   8.173127  0.002827  0.000214    0.002765   
14           8        0  693.4   7.231874  0.007306  0.000196    0.006862   
15           8        1  106.6   7.231874  0.007592  0.000129    0.007543   
16           9        0  674.6  12.361230  0.027144  0.001632    0.024130   
17           9        1  125.4  12.361230  0.026976  0.001635    0.023830   
18          10        0  548.6  11.260551  0.192106  0.013011    0.121512   
19          10        1  251.4  11.260551  0.645223  0.019566    0.374268   

             Mean_FOIF           FOIF_Positive_Fraction           Mean_TracIn  \
         std      mean       std                   mean       std        mean   
0   0.000002  0.000013  0.000002               0.995874  0.002751    1.867631   
1   0.000001  0.000024  0.000009               1.000000  0.000000   -0.008866   
2   0.000005  0.000027  0.000004               0.990198  0.006342    1.861336   
3   0.000008  0.000080  0.000032               1.000000  0.000000   -0.007036   
4   0.000008  0.000039  0.000006               0.976671  0.008422    1.859663   
5   0.000011  0.000138  0.000060               1.000000  0.000000   -0.005189   
6   0.000016  0.000057  0.000007               0.956315  0.032600    1.861733   
7   0.000015  0.000237  0.000092               1.000000  0.000000   -0.006802   
8   0.000025  0.000107  0.000007               0.992667  0.012194    1.854320   
9   0.000028  0.000432  0.000167               1.000000  0.000000   -0.007069   
10  0.000062  0.000267  0.000023               1.000000  0.000000    1.846716   
11  0.000075  0.000887  0.000319               1.000000  0.000000   -0.006824   
12  0.000139  0.000727  0.000139               1.000000  0.000000    1.825163   
13  0.000338  0.002191  0.000783               1.000000  0.000000   -0.008424   
14  0.000123  0.002027  0.000473               0.999710  0.000648    1.817954   
15  0.000152  0.004951  0.001242               1.000000  0.000000   -0.011943   
16  0.001421  0.003312  0.000750               0.985673  0.007973    1.809495   
17  0.001361  0.012328  0.002425               1.000000  0.000000   -0.015712   
18  0.011645 -0.002445  0.000921               0.671344  0.043340    1.796679   
19  0.034256  0.023947  0.004373               0.911212  0.018305   -0.022062   

             TracIn_Positive_Fraction            
         std                     mean       std  
0   0.041623                 1.000000  0.000000  
1   0.002158                 0.144485  0.062942  
2   0.042801                 1.000000  0.000000  
3   0.002010                 0.137320  0.106261  
4   0.043015                 1.000000  0.000000  
5   0.002084                 0.230946  0.106524  
6   0.043578                 1.000000  0.000000  
7   0.003830                 0.182608  0.141813  
8   0.042700             

In [10]:
def mean_std_string(x):
    return f"{x.mean():.6f} ± {x.std(ddof=1):.6f}"


formatted_decile_summary = (
    decile_results_5_runs
    .groupby(
        ["Loss_Decile", "Minority"]
    )[statistics_to_summarise]
    .agg(mean_std_string)
    .reset_index()
)

display(formatted_decile_summary)

,Loss_Decile,Minority,Count,Mean_Loss,Median_Loss,Mean_FOIF,FOIF_Positive_Fraction,Mean_TracIn,TracIn_Positive_Fraction
0,1,0,727.600000 ± 6.542171,0.000035 ± 0.000002,0.000035 ± 0.000002,0.000013 ± 0.000002,0.995874 ± 0.002751,1.867631 ± 0.041623,1.000000 ± 0.000000
1,1,1,73.800000 ± 5.540758,0.000022 ± 0.000001,0.000020 ± 0.000001,0.000024 ± 0.000009,1.000000 ± 0.000000,-0.008866 ± 0.002158,0.144485 ± 0.062942
2,2,0,774.200000 ± 9.471008,0.000080 ± 0.000004,0.000079 ± 0.000005,0.000027 ± 0.000004,0.990198 ± 0.006342,1.861336 ± 0.042801,1.000000 ± 0.000000
3,2,1,25.400000 ± 7.829432,0.000079 ± 0.000005,0.000079 ± 0.000008,0.000080 ± 0.000032,1.000000 ± 0.000000,-0.007036 ± 0.002010,0.137320 ± 0.106261
4,3,0,771.600000 ± 3.847077,0.000141 ± 0.000008,0.000139 ± 0.000008,0.000039 ± 0.000006,0.976671 ± 0.008422,1.859663 ± 0.043015,1.000000 ± 0.000000
5,3,1,27.400000 ± 3.577709,0.000141 ± 0.000008,0.000141 ± 0.000011,0.000138 ± 0.000060,1.000000 ± 0.000000,-0.005189 ± 0.002084,0.230946 ± 0.106524
6,4,0,768.800000 ± 4.604346,0.000251 ± 0.000016,0.000246 ± 0.000016,0.000057 ± 0.000007,0.956315 ± 0.032600,1.861733 ± 0.043578,1.000000 ± 0.000000
7,4,1,31.400000 ± 4.393177,0.000256 ± 0.000013,0.000256 ± 0.000015,0.000237 ± 0.000092,1.000000 ± 0.000000,-0.006802 ± 0.003830,0.182608 ± 0.141813
8,5,0,762.200000 ± 3.420526,0.000485 ± 0.000027,0.000468 ± 0.000025,0.000107 ± 0.000007,0.992667 ± 0.012194,1.854320 ± 0.042700,1.000000 ± 0.000000
9,5,1,37.600000 ± 3.361547,0.000500 ± 0.000030,0.000491 ± 0.000028,0.000432 ± 0.000167,1.000000 ± 0.000000,-0.007069 ± 0.001856,0.141458 ± 0.121288


In [11]:
# density_df["Loss_Decile"] = pd.qcut(
#     density_df["Training_Loss"],
#     q=10,
#     labels=False,
#     duplicates="drop"
# ) + 1

In [12]:
# decile_label_summary = (
#     density_df
#     .groupby(["Loss_Decile", "Minority"])
#     .agg(
#         Count=("Train_ID", "size"),
#         Mean_Loss=("Training_Loss", "mean"),

#         Mean_FOIF=("FOIF_Score", "mean"),
#         FOIF_Positive_Fraction=(
#             "FOIF_Score",
#             lambda x: (x > 0).mean()
#         ),

#         Mean_TracIn=("TracIn_Score", "mean"),
#         TracIn_Positive_Fraction=(
#             "TracIn_Score",
#             lambda x: (x > 0).mean()
#         )
#     )
#     .reset_index()
# )

# print(decile_label_summary)